# Экзаменационное решение: классификация вина по «грязному» датасету

Датасет: `wine_exam_example_dirty.csv`.

Цель работы — построить решение задачи **многоклассовой классификации** по критериям экзамена:

- описать структуру набора данных и целевую переменную;
- выполнить предобработку: дубликаты, пропуски, грязные значения, выбросы, лишние признаки;
- провести визуальный анализ данных;
- построить не менее двух моделей классификации;
- обязательно использовать нейронную модель;
- подобрать гиперпараметры;
- построить кривые обучения и валидации;
- сравнить модели и сохранить итоговую модель.

В этом решении используются две модели:

1. `LogisticRegression` — классическая модель классификации. Это не линейная регрессия, потому что задача не регрессии, а классификации.
2. `MLPClassifier` — нейронная сеть, многослойный перцептрон для табличных данных.

Код специально написан так, чтобы его можно было адаптировать на экзамене под похожий датасет с 3, 5 или другим количеством классов.

# 0. Импорт библиотек

В этом блоке подключаются основные библиотеки:

- `pandas`, `numpy` — работа с таблицами и числами;
- `matplotlib` — графики;
- `sklearn` — предобработка, модели, метрики, подбор гиперпараметров;
- `joblib` — сохранение итоговой модели.

Отдельно импортируются `Pipeline` и `ColumnTransformer`, потому что на экзамене это самый безопасный способ не допустить утечки данных: преобразования обучаются только на train-выборке.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib

RANDOM_STATE = 42

# 1. Загрузка данных и первичный осмотр

На экзамене важно сразу показать, что данные изучены, а не просто загружены.

В этом блоке проверяем:

- размер таблицы;
- первые строки;
- типы данных;
- количество пропусков;
- количество дубликатов;
- потенциальную целевую переменную.

In [ ]:
DATA_PATH = 'wine_exam_example_dirty.csv'

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    df = pd.read_csv('/mnt/data/wine_exam_example_dirty.csv')

print('Размер датасета:', df.shape)
df.head()

In [ ]:
print('Информация о датасете:')
df.info()

In [ ]:
print('Количество пропусков по столбцам:')
df.isna().sum().sort_values(ascending=False)

In [ ]:
print('Количество полных дубликатов:', df.duplicated().sum())
print('Количество уникальных значений по столбцам:')
df.nunique().sort_values()

# 2. Описание структуры датасета и целевой переменной

По смыслу датасета признаки описывают химические характеристики вина, а целевая переменная показывает класс вина.

В этом файле целевая переменная — `WineClass`.

Если на экзамене название target будет другим, достаточно заменить значение переменной `TARGET`.

In [ ]:
TARGET = 'WineClass'

print('Целевая переменная:', TARGET)
print('Классы целевой переменной:')
print(df[TARGET].value_counts().sort_index())

print('\nДоли классов:')
print((df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + '%')

In [ ]:
# Разделим признаки по исходным типам, чтобы понимать структуру таблицы
numeric_cols_initial = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols_initial = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Числовые столбцы:', numeric_cols_initial)
print('Категориальные столбцы:', categorical_cols_initial)

# 3. Предобработка данных

Предобработка закрывает критерий A.

Здесь выполняются действия:

1. удаление дубликатов;
2. очистка грязных категориальных значений;
3. удаление технических и вредных признаков;
4. обработка выбросов методом IQR;
5. подготовка данных к дальнейшему анализу.

Числовые пропуски не заполняются вручную в исходной таблице. Они будут заполнены внутри `Pipeline` медианой, уже после разбиения данных. Так корректнее, потому что модель не получает информацию из validation-выборки.

In [ ]:
clean_df = df.copy()

print('Строк до удаления дубликатов:', clean_df.shape[0])
print('Полных дубликатов:', clean_df.duplicated().sum())

clean_df = clean_df.drop_duplicates()

print('Строк после удаления дубликатов:', clean_df.shape[0])

In [ ]:
# Очистка текстовых категориальных признаков: убираем лишние пробелы и приводим к единому виду.
# Целевую переменную не трогаем, чтобы сохранить исходные названия классов.

for col in clean_df.select_dtypes(include=['object', 'category']).columns:
    if col != TARGET:
        clean_df[col] = (
            clean_df[col]
            .astype('string')
            .str.strip()
            .str.lower()
            .replace({'': np.nan, 'nan': np.nan, 'none': np.nan, '?': np.nan})
        )

# В тестовом датасете в AlcoholLevel специально добавлены грязные значения.
# Допустимыми считаем только low / medium / high.
if 'AlcoholLevel' in clean_df.columns:
    allowed_levels = ['low', 'medium', 'high']
    before_values = clean_df['AlcoholLevel'].value_counts(dropna=False)
    clean_df['AlcoholLevel'] = clean_df['AlcoholLevel'].where(
        clean_df['AlcoholLevel'].isin(allowed_levels),
        np.nan
    )
    after_values = clean_df['AlcoholLevel'].value_counts(dropna=False)
    
    print('AlcoholLevel до очистки:')
    print(before_values)
    print('\nAlcoholLevel после очистки:')
    print(after_values)
else:
    print('Столбец AlcoholLevel отсутствует')

In [ ]:
# Удаляем признаки, которые нельзя использовать при обучении.
# SampleID — технический идентификатор объекта.
# TargetCodeLeak — утечка целевой переменной, так как он напрямую связан с классом вина.

cols_to_drop = []

for col in ['SampleID', 'TargetCodeLeak']:
    if col in clean_df.columns:
        cols_to_drop.append(col)

clean_df = clean_df.drop(columns=cols_to_drop)

print('Удалённые признаки:', cols_to_drop)
print('Размер таблицы после удаления лишних признаков:', clean_df.shape)

## 3.1. Анализ и обработка выбросов

Для числовых признаков используется метод межквартильного размаха IQR:

- нижняя граница: `Q1 - 1.5 * IQR`;
- верхняя граница: `Q3 + 1.5 * IQR`.

Сильно выбивающиеся значения не удаляются, а ограничиваются границами. Это называется clipping/winsorization. Такой подход удобен для экзамена: размер выборки сохраняется, а чрезмерные выбросы меньше влияют на модели.

In [ ]:
def make_iqr_report(data, columns, factor=1.5):
    report = []
    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        outliers = ((data[col] < lower) | (data[col] > upper)).sum()
        report.append({
            'feature': col,
            'q1': q1,
            'q3': q3,
            'iqr': iqr,
            'lower_bound': lower,
            'upper_bound': upper,
            'outliers_count': int(outliers)
        })
    return pd.DataFrame(report)

numeric_cols = clean_df.select_dtypes(include=np.number).columns.tolist()
outlier_report_before = make_iqr_report(clean_df, numeric_cols)

outlier_report_before.sort_values('outliers_count', ascending=False)

In [ ]:
def cap_outliers_iqr(data, columns, factor=1.5):
    result = data.copy()
    limits = {}
    
    for col in columns:
        q1 = result[col].quantile(0.25)
        q3 = result[col].quantile(0.75)
        iqr = q3 - q1
        
        if pd.isna(iqr) or iqr == 0:
            continue
        
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        limits[col] = (lower, upper)
        result[col] = result[col].clip(lower=lower, upper=upper)
    
    return result, limits

clean_df, outlier_limits = cap_outliers_iqr(clean_df, numeric_cols)

outlier_report_after = make_iqr_report(clean_df, numeric_cols)
print('Количество признаков, для которых применялись границы IQR:', len(outlier_limits))
outlier_report_after.sort_values('outliers_count', ascending=False)

In [ ]:
print('Пропуски после очистки грязных значений:')
clean_df.isna().sum().sort_values(ascending=False)

# 4. Разведочный анализ данных — EDA

EDA закрывает критерий B.

Нужно показать не просто графики, а смысл:

- как распределена целевая переменная;
- какие распределения имеют числовые признаки;
- есть ли сильные корреляции;
- есть ли различия признаков между классами;
- как выглядят boxplot-графики после обработки выбросов.

In [ ]:
plt.figure(figsize=(6, 4))
clean_df[TARGET].value_counts().sort_index().plot(kind='bar')
plt.title('Распределение объектов по классам')
plt.xlabel('Класс вина')
plt.ylabel('Количество объектов')
plt.grid(axis='y')
plt.show()

In [ ]:
# Распределения числовых признаков.
# density=True показывает не просто количество объектов, а плотность распределения.

numeric_cols = clean_df.select_dtypes(include=np.number).columns.tolist()

clean_df[numeric_cols].hist(figsize=(14, 10), bins=20, density=True)
plt.suptitle('Плотности распределения числовых признаков')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot по всем числовым признакам после обработки выбросов
plt.figure(figsize=(14, 6))
clean_df[numeric_cols].boxplot(rot=45)
plt.title('Boxplot числовых признаков после обработки выбросов')
plt.ylabel('Значения признаков')
plt.grid(True)
plt.show()

In [ ]:
# Корреляционная матрица числовых признаков
corr = clean_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Корреляционная матрица числовых признаков')
plt.tight_layout()
plt.show()

In [ ]:
# Посмотрим наиболее сильные корреляции между признаками
corr_pairs = (
    corr.abs()
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

corr_pairs.head(10)

In [ ]:
# Влияние признаков на целевую переменную.
# Для каждого класса считаем средние значения числовых признаков.
class_means = clean_df.groupby(TARGET)[numeric_cols].mean()

# Чтобы график не был слишком перегруженным, возьмём несколько признаков с заметной изменчивостью между классами.
selected_features = class_means.var(axis=0).sort_values(ascending=False).head(5).index.tolist()

class_means[selected_features].plot(kind='bar', figsize=(10, 5))
plt.title('Средние значения наиболее различающихся признаков по классам')
plt.xlabel('Класс вина')
plt.ylabel('Среднее значение признака')
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.show()

class_means[selected_features]

In [ ]:
# Boxplot нескольких важных признаков по классам.
# Такие графики хорошо показывают, разделяет ли признак классы.

for feature in selected_features[:3]:
    plt.figure(figsize=(7, 5))
    clean_df.boxplot(column=feature, by=TARGET, grid=True)
    plt.title(f'Распределение {feature} по классам')
    plt.suptitle('')
    plt.xlabel('Класс вина')
    plt.ylabel(feature)
    plt.show()

## Краткий вывод по EDA

По графикам нужно написать несколько предложений своими словами.

Пример вывода:

- классы в датасете представлены не одинаково, поэтому при разбиении нужно использовать `stratify`;
- числовые признаки имеют разные масштабы, поэтому для логистической регрессии и нейронной сети нужна стандартизация;
- часть признаков заметно отличается по классам, значит задача классификации имеет смысл;
- корреляционная матрица позволяет увидеть признаки, которые связаны между собой;
- boxplot показывает, что после обработки сильные выбросы ограничены.

# 5. Формирование признаков и целевой переменной

На этом этапе формируются:

- `X` — матрица признаков;
- `y` — целевая переменная.

Целевую переменную кодируем через `LabelEncoder`, потому что модели работают с числовыми метками классов.

Код автоматически подхватит любое количество классов. Поэтому если на экзамене будет не 3, а 5 категорий, этот блок всё равно будет работать.

In [ ]:
X = clean_df.drop(columns=[TARGET])
y = clean_df[TARGET]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print('Классы:', list(label_encoder.classes_))
print('Закодированные классы:', sorted(np.unique(y_encoded)))
print('Размер X:', X.shape)
print('Размер y:', y_encoded.shape)

# 6. Разбиение на обучающую и валидационную выборки

Используется `stratify=y_encoded`, чтобы сохранить примерно такое же соотношение классов в train и validation.

Это важно при несбалансированной классификации: иначе один из классов может быть плохо представлен в validation-выборке, и оценка качества будет ненадёжной.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_encoded,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print('Размер обучающей выборки:', X_train.shape)
print('Размер валидационной выборки:', X_valid.shape)

print('\nРаспределение классов в train:')
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print('\nРаспределение классов в validation:')
print(pd.Series(y_valid).value_counts(normalize=True).sort_index())

# 7. Pipeline предобработки

В этом блоке задаётся универсальная предобработка признаков.

Для числовых признаков:

- пропуски заполняются медианой;
- признаки стандартизируются через `StandardScaler`.

Для категориальных признаков:

- пропуски заполняются самым частым значением;
- категории кодируются через `OneHotEncoder`.

Такой подход особенно удобен на экзамене: один и тот же `preprocessor` можно использовать для обеих моделей.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Числовые признаки:', numeric_features)
print('Категориальные признаки:', categorical_features)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Разные версии sklearn используют разные названия параметра для плотного one-hot результата.
try:
    one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', one_hot_encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Проверка: после применения preprocessor к train не должно остаться пропусков.
X_train_prepared = preprocessor.fit_transform(X_train)

print('Размер подготовленной train-матрицы:', X_train_prepared.shape)
print('Количество NaN после предобработки:', np.isnan(X_train_prepared).sum())

# 8. Метрика качества

Для многоклассовой классификации будем смотреть несколько метрик:

- `accuracy` — общая доля правильных ответов;
- `precision`, `recall`, `f1-score` по каждому классу;
- `f1_macro` — среднее F1 по классам без учёта их размера.

Основной метрикой выберем `f1_macro`, потому что она лучше подходит, если классы представлены неравномерно.

In [ ]:
def evaluate_model(model_name, model, X_valid, y_valid, label_encoder):
    y_pred = model.predict(X_valid)
    accuracy = accuracy_score(y_valid, y_pred)
    f1_macro = f1_score(y_valid, y_pred, average='macro')
    
    print(f'Модель: {model_name}')
    print('Accuracy:', round(accuracy, 4))
    print('F1 macro:', round(f1_macro, 4))
    print('\nClassification report:')
    print(classification_report(y_valid, y_pred, target_names=label_encoder.classes_))
    
    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'y_pred': y_pred
    }

# 9. Модель 1 — логистическая регрессия

Логистическая регрессия выбрана как базовая классическая модель классификации.

Почему она подходит:

- быстро обучается;
- хорошо работает после стандартизации признаков;
- подходит для многоклассовой классификации;
- даёт понятный baseline, с которым можно сравнить нейронную сеть.

In [ ]:
log_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

log_reg.fit(X_train, y_train)
result_lr = evaluate_model('Logistic Regression', log_reg, X_valid, y_valid, label_encoder)

# 10. Оптимизация логистической регрессии

Для настройки логистической регрессии подбирается параметр `C`.

- маленькое `C` — сильная регуляризация, модель проще;
- большое `C` — слабая регуляризация, модель сложнее.

Поиск выполняется через `GridSearchCV` с кросс-валидацией. В качестве метрики используется `f1_macro`.

In [ ]:
param_grid_lr = {
    'model__C': [0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs']
}

grid_lr = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_lr,
    cv=3,
    scoring='f1_macro',
    n_jobs=1
)

grid_lr.fit(X_train, y_train)

print('Лучшие параметры Logistic Regression:')
print(grid_lr.best_params_)
print('Лучший F1 macro на CV:', round(grid_lr.best_score_, 4))

result_lr_opt = evaluate_model('Optimized Logistic Regression', grid_lr.best_estimator_, X_valid, y_valid, label_encoder)

# 11. Модель 2 — нейронная сеть MLPClassifier

`MLPClassifier` — это полносвязная нейронная сеть.

Почему она подходит:

- работает с табличными признаками;
- может находить нелинейные зависимости;
- закрывает требование экзамена про нейронную модель;
- не требует писать отдельный код на TensorFlow/Keras.

Используется `early_stopping=True`, чтобы модель могла остановиться раньше и меньше переобучалась.

In [ ]:
mlp = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

mlp.fit(X_train, y_train)
result_mlp = evaluate_model('MLPClassifier', mlp, X_valid, y_valid, label_encoder)

# 12. Оптимизация нейронной сети

Для нейронной сети подбираются:

- `hidden_layer_sizes` — число нейронов и слоёв;
- `alpha` — L2-регуляризация;
- `learning_rate_init` — начальная скорость обучения.

Сетка специально небольшая, чтобы код не работал слишком долго на экзамене.

In [ ]:
param_grid_mlp = {
    'model__hidden_layer_sizes': [(32,), (64, 32)],
    'model__alpha': [0.0001, 0.001],
    'model__learning_rate_init': [0.001]
}

grid_mlp = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid_mlp,
    cv=2,
    scoring='f1_macro',
    n_jobs=1
)

grid_mlp.fit(X_train, y_train)

print('Лучшие параметры MLPClassifier:')
print(grid_mlp.best_params_)
print('Лучший F1 macro на CV:', round(grid_mlp.best_score_, 4))

result_mlp_opt = evaluate_model('Optimized MLPClassifier', grid_mlp.best_estimator_, X_valid, y_valid, label_encoder)

# 13. Матрицы ошибок

Матрица ошибок показывает, какие классы модель путает между собой.

Для экзамена это хороший график, потому что он объясняет качество модели не одной цифрой, а по каждому классу.

In [ ]:
for title, y_pred in [
    ('Optimized Logistic Regression', result_lr_opt['y_pred']),
    ('Optimized MLPClassifier', result_mlp_opt['y_pred'])
]:
    cm = confusion_matrix(y_valid, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_encoder.classes_
    )
    disp.plot()
    plt.title(f'Матрица ошибок: {title}')
    plt.show()

# 14. Кривая валидации

Кривая валидации показывает, как качество меняется при разных значениях гиперпараметра.

Здесь рассматривается параметр `alpha` у нейронной сети:

- если train-качество высокое, а validation низкое — есть переобучение;
- если оба качества низкие — есть недообучение;
- если train и validation близки и высокие — модель подобрана удачно.

In [ ]:
param_range_alpha = np.array([0.0001, 0.001, 0.01])

train_scores, valid_scores = validation_curve(
    estimator=Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE
        ))
    ]),
    X=X_train,
    y=y_train,
    param_name='model__alpha',
    param_range=param_range_alpha,
    cv=2,
    scoring='f1_macro',
    n_jobs=1
)

plt.figure(figsize=(8, 5))
plt.semilogx(param_range_alpha, train_scores.mean(axis=1), marker='o', label='Train')
plt.semilogx(param_range_alpha, valid_scores.mean(axis=1), marker='o', label='Validation')
plt.title('Кривая валидации MLPClassifier по alpha')
plt.xlabel('alpha')
plt.ylabel('F1 macro')
plt.legend()
plt.grid(True)
plt.show()

# 15. Кривая обучения

Кривая обучения показывает, как качество меняется при увеличении размера обучающей выборки.

Она помогает сделать вывод о переобучении или недообучении:

- большой разрыв между train и validation — вероятно, переобучение;
- низкие обе кривые — вероятно, недообучение;
- близкие и высокие кривые — модель обучается стабильно.

In [ ]:
train_sizes, train_scores_lc, valid_scores_lc = learning_curve(
    estimator=grid_mlp.best_estimator_,
    X=X_train,
    y=y_train,
    cv=2,
    scoring='f1_macro',
    n_jobs=1,
    train_sizes=np.linspace(0.3, 1.0, 4)
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores_lc.mean(axis=1), marker='o', label='Train')
plt.plot(train_sizes, valid_scores_lc.mean(axis=1), marker='o', label='Validation')
plt.title('Кривая обучения лучшей MLPClassifier')
plt.xlabel('Размер обучающей выборки')
plt.ylabel('F1 macro')
plt.legend()
plt.grid(True)
plt.show()

# 16. График функции потерь нейронной сети

`loss_curve_` показывает, как ошибка нейронной сети менялась по эпохам.

Если loss постепенно уменьшается, значит обучение идёт нормально. Если loss скачет или не уменьшается, могут быть проблемы со скоростью обучения, архитектурой или масштабированием признаков.

In [ ]:
best_mlp_model = grid_mlp.best_estimator_.named_steps['model']

plt.figure(figsize=(8, 5))
plt.plot(best_mlp_model.loss_curve_)
plt.title('График функции потерь лучшей MLPClassifier')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

# 17. Сравнение моделей и выбор лучшей

Сравниваем модели по двум метрикам:

- `accuracy` — общая точность;
- `f1_macro` — основная метрика для выбора модели.

Лучшей считаем модель с максимальным `f1_macro` на validation-выборке.

In [ ]:
results = pd.DataFrame([
    {k: v for k, v in result_lr.items() if k != 'y_pred'},
    {k: v for k, v in result_lr_opt.items() if k != 'y_pred'},
    {k: v for k, v in result_mlp.items() if k != 'y_pred'},
    {k: v for k, v in result_mlp_opt.items() if k != 'y_pred'},
])

results = results.sort_values('f1_macro', ascending=False).reset_index(drop=True)
results

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(results['model_name'], results['f1_macro'])
plt.title('Сравнение моделей по F1 macro')
plt.xlabel('Модель')
plt.ylabel('F1 macro')
plt.xticks(rotation=25, ha='right')
plt.ylim(0, 1)
plt.grid(axis='y')
plt.show()

In [ ]:
if result_mlp_opt['f1_macro'] >= result_lr_opt['f1_macro']:
    best_model = grid_mlp.best_estimator_
    best_model_name = 'optimized_mlp_classifier'
    best_score = result_mlp_opt['f1_macro']
else:
    best_model = grid_lr.best_estimator_
    best_model_name = 'optimized_logistic_regression'
    best_score = result_lr_opt['f1_macro']

print('Лучшая модель:', best_model_name)
print('F1 macro лучшей модели:', round(best_score, 4))

# 18. Сохранение итоговой модели

Сохраняем:

- лучшую модель вместе со всей предобработкой;
- `LabelEncoder`, чтобы потом переводить числовые предсказания обратно в названия классов.

Это закрывает критерий сохранения итоговой модели для дальнейшего использования.

In [ ]:
joblib.dump(best_model, 'best_exam_model.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

print('Модель сохранена в файл best_exam_model.pkl')
print('Кодировщик классов сохранён в файл label_encoder.pkl')

In [ ]:
# Проверка загрузки модели и получения предсказаний
loaded_model = joblib.load('best_exam_model.pkl')
loaded_encoder = joblib.load('label_encoder.pkl')

sample_pred_encoded = loaded_model.predict(X_valid.head(5))
sample_pred_labels = loaded_encoder.inverse_transform(sample_pred_encoded)

print('Предсказания для первых 5 объектов validation:')
print(sample_pred_labels)

# 19. Итоговый вывод для отчёта

В работе была решена задача многоклассовой классификации вина.

Была выполнена предобработка данных: удалены дубликаты, очищен категориальный признак `AlcoholLevel`, исключены технический идентификатор `SampleID` и признак `TargetCodeLeak`, являющийся утечкой целевой переменной. Для числовых признаков была проведена обработка выбросов методом IQR. Пропущенные значения обрабатывались внутри `Pipeline`: числовые признаки заполнялись медианой, категориальные — самым частым значением.

На этапе EDA были построены графики распределения целевой переменной, плотности распределения числовых признаков, boxplot-графики, корреляционная матрица и визуализация различий признаков между классами. Было показано, что признаки имеют разные масштабы и распределения, поэтому стандартизация необходима для логистической регрессии и нейронной сети.

Для прогнозирования были построены две модели: логистическая регрессия и нейронная сеть `MLPClassifier`. Для обеих моделей выполнен подбор гиперпараметров через `GridSearchCV`. Качество оценивалось с помощью `accuracy`, `classification_report` и основной метрики `f1_macro`, подходящей для многоклассовой классификации.

Дополнительно были построены матрицы ошибок, кривая валидации, кривая обучения и график функции потерь нейронной сети. По результатам сравнения была выбрана лучшая модель по `f1_macro`, после чего она была сохранена в файл для дальнейшего использования.

# 20. Мини-шпаргалка для экзамена

Если на экзамене будет похожий датасет, порядок действий такой:

1. `pd.read_csv()` — загрузить файл.
2. `df.info()`, `df.isna().sum()`, `df.duplicated().sum()` — проверить структуру и грязь.
3. Найти target и проверить `value_counts()`.
4. Удалить `id`-столбцы и признаки-утечки.
5. Очистить грязные категории.
6. Проверить и обработать выбросы.
7. Построить EDA: target, hist/density, boxplot, correlation, влияние признаков на target.
8. Сделать `LabelEncoder` для target.
9. Сделать `train_test_split(..., stratify=y)`.
10. Собрать `ColumnTransformer` + `Pipeline`.
11. Обучить классику: `LogisticRegression`.
12. Обучить нейронку: `MLPClassifier`.
13. Сделать `GridSearchCV`.
14. Вывести `classification_report`, confusion matrix, learning/validation curves.
15. Сравнить модели по `f1_macro`.
16. Сохранить модель через `joblib.dump()`.